In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s5e10/sample_submission.csv
/kaggle/input/playground-series-s5e10/train.csv
/kaggle/input/playground-series-s5e10/test.csv
/kaggle/input/pss5e10-main/__results__.html
/kaggle/input/pss5e10-main/xgb_5_fold_te_log_transformed.csv
/kaggle/input/pss5e10-main/__notebook__.ipynb
/kaggle/input/pss5e10-main/__output__.json
/kaggle/input/pss5e10-main/Train_org.csv
/kaggle/input/pss5e10-main/test.csv
/kaggle/input/pss5e10-main/final_train.csv
/kaggle/input/pss5e10-main/custom.css
/kaggle/input/pss5e10-main/__results___files/__results___11_2.png


In [2]:
!pip install autogluon.tabular[0]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.3/487.3 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.0/278.0 kB 18.0 MB/s eta 0:00:00
  Attempting uninstall: psutil
    Found existing installation: psutil 7.1.0
    Uninstalling psutil-7.1.0:
      Successfully uninstalled psutil-7.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.2
    Uninstalling scikit-learn-1.2.2:
      Successfully uninstalled scikit-learn-1.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
category-encoders 2.7.

In [3]:
train = pd.read_csv('/kaggle/input/pss5e10-main/final_train.csv')
# train = train.fillna(0)
test = pd.read_csv('/kaggle/input/pss5e10-main/test.csv')
test = test.drop(columns='accident_risk')

In [4]:
TARGET = 'accident_risk'
FEATURES = [col for col in train.columns if col!='accident_risk']

In [5]:
train = train.fillna(0)
test = test.fillna(0)

In [6]:
import shutil, os
old = '/kaggle/working/AutogluonModels'
if os.path.exists(old):
    shutil.rmtree(old)

In [7]:
from autogluon.tabular import TabularPredictor
import warnings
warnings.filterwarnings('ignore')
predictor = TabularPredictor(label=TARGET, eval_metric='rmse', problem_type='regression', path=f'AutogluonModels/run_0')

predictor.fit(
    train_data=train,
    time_limit=18000,
    presets='best_quality',
    num_bag_folds=7,
    num_stack_levels=3,         # Deeper stacking
    num_bag_sets=3,            # More ensembles
    auto_stack=True,
    raise_on_no_models_fitted=False,
    hyperparameters={
        'GBM': {'tree_method': 'gpu_hist', 'device': 'gpu'},  # LightGBM
        'CAT': {'task_type': 'GPU'},  # CatBoost
        'XGB': {'tree_method': 'gpu_hist', 'gpu_id': 0},  # XGBoost
        'NN_TORCH': {},  # Neural networks with GPU
        'RF': {},         # Random Forest
        'XT': {},         # Extra Trees
        'LR': {},          # Linear Regression
        # You can add others, e.g., KNN if desired:
        'KNN': {}
    },
    verbosity=1,
    num_gpus=1
)

Will use sequential fold fitting strategy because import of ray failed. Reason: ray==2.49.2 detected. 2.10.0 <= ray < 2.45.0 is required. You can use pip to install certain version of ray `pip install "ray>=2.10.0,<2.45.0"`
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
		[22:17:53] /workspace/src/context.cc:173: Both `device` and `gpu_id` are specifie

In [8]:
leaderboard = predictor.leaderboard()
leaderboard

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L5,-0.055721,root_mean_squared_error,372.816844,12447.351751,0.009577,1.558618,5,True,27
1,WeightedEnsemble_L4,-0.055730,root_mean_squared_error,308.879097,8966.304822,0.008556,0.383092,4,True,22
2,LinearModel_BAG_L3,-0.055733,root_mean_squared_error,279.643354,7643.643393,1.725674,45.632042,3,True,20
3,WeightedEnsemble_L3,-0.055739,root_mean_squared_error,260.029963,5349.667846,0.011952,0.403231,3,True,15
4,ExtraTrees_BAG_L2,-0.055748,root_mean_squared_error,231.066786,4052.786009,24.192274,362.688921,2,True,12
5,LightGBM_BAG_L4,-0.055764,root_mean_squared_error,349.686334,11185.928232,5.646307,77.614347,4,True,23
6,LightGBM_BAG_L3,-0.055772,root_mean_squared_error,284.096775,7678.957417,6.179094,80.946066,3,True,16
7,CatBoost_BAG_L3,-0.055838,root_mean_squared_error,278.676356,7679.543829,0.758675,81.532478,3,True,18
8,ExtraTrees_BAG_L3,-0.055880,root_mean_squared_error,302.146615,7949.665783,24.228935,351.654432,3,True,19
9,ExtraTrees_BAG_L4,-0.055883,root_mean_squared_error,368.268209,11468.078572,24.228182,359.764687,4,True,26


In [9]:
models = leaderboard['model'].to_list()
oofs_dict = {}
for m in models:
    oofs_dict[m] = predictor.predict_oof(model=m)

oofs_df = pd.DataFrame(oofs_dict)

In [10]:
predictor2 = TabularPredictor.load('/kaggle/working/AutogluonModels/run_0')
test_preds = {}
for m in models:
    test_preds[m] = predictor2.predict(test, model=m)

In [11]:
test_preds_df = pd.DataFrame(test_preds)
test_preds_df

,WeightedEnsemble_L5,WeightedEnsemble_L4,LinearModel_BAG_L3,WeightedEnsemble_L3,ExtraTrees_BAG_L2,LightGBM_BAG_L4,LightGBM_BAG_L3,CatBoost_BAG_L3,ExtraTrees_BAG_L3,ExtraTrees_BAG_L4,...,CatBoost_BAG_L1,RandomForest_BAG_L4,RandomForest_BAG_L3,ExtraTrees_BAG_L1,RandomForest_BAG_L1,NeuralNetTorch_BAG_L3,NeuralNetTorch_BAG_L2,NeuralNetTorch_BAG_L1,LinearModel_BAG_L1,KNeighbors_BAG_L1
0,0.289724,0.289814,0.290577,0.291346,0.292115,0.289796,0.293204,0.295002,0.286311,0.288261,...,0.295211,0.286699,0.282947,0.303857,0.302449,0.305633,0.306763,0.304732,0.302729,0.272
1,0.123218,0.123536,0.123639,0.123700,0.122975,0.123327,0.124758,0.123274,0.124396,0.122913,...,0.122818,0.121899,0.122611,0.120544,0.120227,0.132217,0.128524,0.134027,0.117168,0.088
2,0.181192,0.180742,0.180337,0.179982,0.179918,0.182664,0.180994,0.181952,0.182167,0.181786,...,0.187472,0.180238,0.184388,0.176162,0.176671,0.192775,0.197914,0.200865,0.190230,0.190
3,0.311128,0.310096,0.310015,0.311893,0.312314,0.311912,0.312004,0.309680,0.310066,0.312232,...,0.314187,0.312989,0.310823,0.317455,0.317287,0.322465,0.323937,0.322732,0.393189,0.354
4,0.403352,0.405008,0.404989,0.406270,0.406783,0.402853,0.403108,0.400996,0.404774,0.402834,...,0.392639,0.397922,0.405186,0.408725,0.409216,0.387531,0.393841,0.393668,0.366800,0.388
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172580,0.103058,0.103296,0.103488,0.102774,0.102145,0.101709,0.102554,0.101430,0.105601,0.104634,...,0.105168,0.100882,0.101573,0.102798,0.103341,0.113837,0.115308,0.120876,0.106855,0.086
172581,0.518919,0.520607,0.520644,0.520629,0.520981,0.518759,0.519498,0.520677,0.516753,0.513033,...,0.518180,0.511594,0.520271,0.507989,0.506931,0.513642,0.507836,0.514133,0.514560,0.470
172582,0.247146,0.247034,0.247058,0.246768,0.246785,0.247949,0.247402,0.248533,0.246610,0.242323,...,0.249810,0.246910,0.246825,0.238420,0.240877,0.255581,0.259682,0.260972,0.240363,0.282
172583,0.126585,0.127103,0.127276,0.127575,0.126733,0.126055,0.127080,0.128797,0.126679,0.125870,...,0.128154,0.123707,0.125538,0.126076,0.126277,0.136505,0.139199,0.142940,0.128108,0.160


In [12]:
samp = pd.read_csv('/kaggle/input/playground-series-s5e10/sample_submission.csv')
samp['accident_risk'] = test_preds_df.iloc[:,0]
samp.to_csv('sample_submission.csv', index=False)

In [13]:
leaderboard.to_csv('leaderboard_autogluon_baseline.csv', index=False)
oofs_df.to_csv('oofs_autogluon_baseline.csv', index=False)
test_preds_df.to_csv('test_preds_autogluon_baseline', index=False)